In [21]:
!pip install -q chromadb pypdf requests

In [20]:
import chromadb
import requests
import os
import subprocess
import time
from pypdf import PdfReader

print("ChromaDB version:", chromadb.__version__)
print("Libraries imported successfully!")

ChromaDB version: 1.5.9
Libraries imported successfully!


In [22]:
client = chromadb.Client()

print("ChromaDB client created successfully!")

ChromaDB client created successfully!


In [23]:
collection = client.get_or_create_collection(
    name="w5d3_documents",
    metadata={"hnsw:space": "cosine"}
)

print("Collection created:", collection.name)

Collection created: w5d3_documents


In [24]:
documents = [
    "Machine learning enables computers to learn patterns from data.",
    "Supervised learning uses labeled training data.",
    "Unsupervised learning discovers patterns in unlabeled data.",
    "Deep learning uses neural networks with multiple layers.",
    "Neural networks are inspired by the structure of the human brain.",
    "Classification predicts categories or classes.",
    "Regression predicts continuous numerical values.",
    "Overfitting happens when a model learns training data too closely.",
    "Underfitting occurs when a model is too simple to learn patterns.",
    "Cross validation helps evaluate machine learning models reliably.",
    "Feature engineering transforms raw data into useful features.",
    "Standardization scales features around a mean of zero.",
    "Decision trees make predictions using a sequence of rules.",
    "Random forests combine multiple decision trees.",
    "Support vector machines find useful decision boundaries.",
    "K nearest neighbors predicts using nearby data points.",
    "Clustering groups similar data points together.",
    "Natural language processing enables computers to process human language.",
    "Large language models can generate and understand text.",
    "Retrieval augmented generation combines retrieval with language generation."
]

print("Number of documents:", len(documents))

Number of documents: 20


In [25]:
ids = [f"doc_{i}" for i in range(1, 21)]

metadata = [
    {
        "id": i,
        "topic": "AI/ML",
        "source": "W5D3"
    }
    for i in range(1, 21)
]

print("IDs and metadata created!")

IDs and metadata created!


In [26]:
collection.add(
    documents=documents,
    metadatas=metadata,
    ids=ids
)

print("20 documents added successfully!")

20 documents added successfully!


In [27]:
print("Total documents:", collection.count())

result = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

for doc, meta in zip(result["documents"], result["metadatas"]):
    print(meta["id"], ":", doc)

Total documents: 20
1 : Machine learning enables computers to learn patterns from data.
2 : Supervised learning uses labeled training data.
3 : Unsupervised learning discovers patterns in unlabeled data.
4 : Deep learning uses neural networks with multiple layers.
5 : Neural networks are inspired by the structure of the human brain.


In [28]:
query = "How does machine learning learn from data?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 results:")

for i, doc in enumerate(results["documents"][0]):
    print(f"\n{i+1}. {doc}")
    print("Distance:", results["distances"][0][i])

Query: How does machine learning learn from data?

Top 3 results:

1. Machine learning enables computers to learn patterns from data.
Distance: 0.2601284384727478

2. Supervised learning uses labeled training data.
Distance: 0.4293709397315979

3. Cross validation helps evaluate machine learning models reliably.
Distance: 0.4830223321914673


In [29]:
filtered_results = collection.get(
    where={"topic": "AI/ML"},
    include=["documents", "metadatas"]
)

print("Documents with topic = AI/ML:")
print("Number of results:", len(filtered_results["documents"]))

for doc in filtered_results["documents"][:5]:
    print("-", doc)

Documents with topic = AI/ML:
Number of results: 20
- Machine learning enables computers to learn patterns from data.
- Supervised learning uses labeled training data.
- Unsupervised learning discovers patterns in unlabeled data.
- Deep learning uses neural networks with multiple layers.
- Neural networks are inspired by the structure of the human brain.


In [30]:
print("Manual verification:")
print("Query: How does machine learning learn from data?")
print("\nThe retrieved results should mainly discuss:")
print("- Machine learning")
print("- Learning patterns from data")
print("- Supervised/unsupervised learning")

Manual verification:
Query: How does machine learning learn from data?

The retrieved results should mainly discuss:
- Machine learning
- Learning patterns from data
- Supervised/unsupervised learning


In [32]:
from google.colab import files

uploaded = files.upload()

Saving W5D3_AI_ML_Retrieval_Document.pdf to W5D3_AI_ML_Retrieval_Document.pdf


In [33]:
import os

pdf_files = [f for f in os.listdir("/content") if f.lower().endswith(".pdf")]

print("PDF files found:", pdf_files)

PDF files found: ['W5D3_AI_ML_Retrieval_Document.pdf']


In [34]:
pdf_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".pdf")
]

print("PDF files found:", pdf_files)

PDF files found: ['W5D3_AI_ML_Retrieval_Document.pdf']


In [35]:
pdf_path = "/content/" + pdf_files[0]

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("PDF loaded successfully!")
print("Pages:", len(reader.pages))
print("Characters extracted:", len(text))

PDF loaded successfully!
Pages: 2
Characters extracted: 3589


In [36]:
chunk_size = 800
overlap = 100

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0][:1000])

Number of chunks: 6

First chunk:

AI and Machine Learning: A Short Reference Document
1. Machine Learning
Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and
make predictions or decisions without being explicitly programmed for every situation. A machine learning
workflow usually includes collecting data, preprocessing it, selecting useful features, training a model,
evaluating the model, and deploying it.
2. Supervised Learning
Supervised learning uses labeled training data. Each training example contains input features and a known
target value. Classification is used when the target represents a category, such as spam or not spam.
Regression is used when the target is a continuous numerical value, such as house price.
3. Unsupervised Learning
Unsupervised learnin


In [37]:
pdf_collection = client.get_or_create_collection(
    name="w5d3_pdf_chunks",
    metadata={"hnsw:space": "cosine"}
)

chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

chunk_metadata = [
    {
        "source": pdf_files[0],
        "chunk": i
    }
    for i in range(len(chunks))
]

pdf_collection.add(
    documents=chunks,
    metadatas=chunk_metadata,
    ids=chunk_ids
)

print("PDF chunks added to ChromaDB!")
print("Total chunks:", pdf_collection.count())

PDF chunks added to ChromaDB!
Total chunks: 6


In [38]:
query = "What is the main topic discussed in this document?"

pdf_results = pdf_collection.query(
    query_texts=[query],
    n_results=3
)

print("Top 3 retrieved chunks:\n")

for i, chunk in enumerate(pdf_results["documents"][0]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:1000])

Top 3 retrieved chunks:


--- Chunk 1 ---
e compared for similarity. ChromaDB is a
vector database commonly used for storing documents and retrieving relevant information. Similarity search
can identify documents or chunks that are semantically related to a query.
10. Large Language Models
Large language models, or LLMs, are neural network models trained on large collections of text. They can
generate, summarize, classify, and answer questions about text. In a RAG system, an LLM can use retrieved
document chunks as context before generating an answer.
Purpose of this document
This document is provided as a sample PDF for the W5D3 ChromaDB practical. It can be used to
demonstrate PDF text extraction, document chunking, vector storage, similarity retrieval, and passing
retrieved context to a local Ollama language model.



--- Chunk 2 ---
ew data. Underfitting occurs when a model is too simple to capture important patterns. Regularization,
cross-validation, and appropriate model complexi

In [40]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 86 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (441 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [41]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [42]:
!ollama --version

In [43]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)
print("Ollama server started!")

Ollama server started!


In [44]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)
print(response.text)

Status: 200
{"models":[]}


In [45]:
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

response = requests.get("http://localhost:11434/api/tags")

print("Ollama status:", response.status_code)
print(response.text[:500])

Ollama status: 200
{"models":[]}


In [46]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [47]:
!ollama pull llama3.2:3b

In [48]:
context = "\n\n".join(pdf_results["documents"][0])

prompt = f"""
Answer the question using only the context provided below.

Context:
{context}

Question:
What is the main topic discussed in this document?

Give a clear and concise answer.
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": prompt,
        "stream": False
    }
)

result = response.json()

print("Ollama answer:")
print(result["response"])

Ollama answer:
The main topic discussed in this document is Retrieval-Augmented Generation (RAG) systems, specifically how they combine information retrieval with a language model to answer questions, using vector databases like ChromaDB for document chunking and storage.


In [49]:
print("===== W5D3 RAG VERIFICATION =====")

print("\nPDF:")
print(pdf_files[0])

print("\nChunks stored in ChromaDB:")
print(pdf_collection.count())

print("\nRetrieved chunks:")
print(len(pdf_results["documents"][0]))

print("\nFinal LLM Answer:")
print(result["response"])

print("\nRAG pipeline completed successfully!")

===== W5D3 RAG VERIFICATION =====

PDF:
W5D3_AI_ML_Retrieval_Document.pdf

Chunks stored in ChromaDB:
6

Retrieved chunks:
3

Final LLM Answer:
The main topic discussed in this document is Retrieval-Augmented Generation (RAG) systems, specifically how they combine information retrieval with a language model to answer questions, using vector databases like ChromaDB for document chunking and storage.

RAG pipeline completed successfully!
